# Impacts of Normalization Techniques

### Loading the Data of Notebook 1

In [2]:
import pickle
import numpy as np

DATA_PATHS = {
    "zscore_all": {
        "train": "./final_split_data_ZscoreAll/train_set.pkl",
        "val":   "./final_split_data_ZscoreAll/val_set.pkl",
        "test":  "./final_split_data_ZscoreAll/test_set.pkl",
    },
    "minmax_all": {
        "train": "./final_split_data_MinMaxAll/train_set.pkl",
        "val":   "./final_split_data_MinMaxAll/val_set.pkl",
        "test":  "./final_split_data_MinMaxAll/test_set.pkl",
    },
    "log_all": {
        "train": "./final_split_data_LogAll/train_set.pkl",
        "val":   "./final_split_data_LogAll/val_set.pkl",
        "test":  "./final_split_data_LogAll/test_set.pkl",
    },
    "hybrid": {
        "train": "./final_split_data_HybridNorm/train_set.pkl",
        "val":   "./final_split_data_HybridNorm/val_set.pkl",
        "test":  "./final_split_data_HybridNorm/test_set.pkl",
    },
}


def load_split(path):
    with open(path, "rb") as f:
        data = pickle.load(f)

    X = data["X"].astype(np.float32)
    y = data["y"].astype(int)

    return X, y


datasets = {}

for norm_name, splits in DATA_PATHS.items():
    X_train, y_train = load_split(splits["train"])
    X_val,   y_val   = load_split(splits["val"])
    X_test,  y_test  = load_split(splits["test"])

    datasets[norm_name] = {
        "X_train": X_train,
        "y_train": y_train,
        "X_val":   X_val,
        "y_val":   y_val,
        "X_test":  X_test,
        "y_test":  y_test,
    }


# ── Summary ────────────────────────────────────────────────────
print(
    f"{'Norm':<14} "
    f"{'Train shape':<22} "
    f"{'Val shape':<22} "
    f"{'Test shape':<22} "
    f"{'Class 0':>8} "
    f"{'Class 1':>8} "
    f"{'Imbalance':>10}"
)
print("─" * 112)

for norm_name, d in datasets.items():
    counts = np.bincount(d["y_train"], minlength=2)

    class_0 = counts[0]
    class_1 = counts[1]
    ratio = class_0 / class_1 if class_1 > 0 else np.inf

    print(
        f"{norm_name:<14} "
        f"{str(d['X_train'].shape):<22} "
        f"{str(d['X_val'].shape):<22} "
        f"{str(d['X_test'].shape):<22} "
        f"{class_0:>8} "
        f"{class_1:>8} "
        f"{ratio:>9.1f}x"
    )


# ── Class imbalance message ────────────────────────────────────
first_key = list(datasets.keys())[0]
counts = np.bincount(datasets[first_key]["y_train"], minlength=2)
ratio = counts[0] / counts[1]

print(f"\nSevere class imbalance detected (~{ratio:.0f}:1).")
print("Consider using class_weight='balanced' in SVM.")
print("For GRU, consider class weights such as:")
print(f"class_weight = {{0: 1.0, 1: {ratio:.1f}}}")

Norm           Train shape            Val shape              Test shape              Class 0  Class 1  Imbalance
────────────────────────────────────────────────────────────────────────────────────────────────────────────────
zscore_all     (12455, 288, 10)       (1780, 288, 10)        (3559, 288, 10)           12337      118     104.6x
minmax_all     (12455, 288, 10)       (1780, 288, 10)        (3559, 288, 10)           12337      118     104.6x
log_all        (12455, 288, 10)       (1780, 288, 10)        (3559, 288, 10)           12337      118     104.6x
hybrid         (12455, 288, 10)       (1780, 288, 10)        (3559, 288, 10)           12337      118     104.6x

Severe class imbalance detected (~105:1).
Consider using class_weight='balanced' in SVM.
For GRU, consider class weights such as:
class_weight = {0: 1.0, 1: 104.6}


### Catch22 Vectorization for SVM

In [3]:
import numpy as np
from joblib import Parallel, delayed
from sktime.transformations.panel.catch22 import Catch22
from sklearn.impute import SimpleImputer
import warnings

# ══════════════════════════════════════════════════════════════
# CATCH22 FEATURE EXTRACTION
# Uses datasets:
# no_norm, zscore_all, minmax_all, log_all, hybrid
# Keeps results only in memory as datasets_svm
# ══════════════════════════════════════════════════════════════

def _catch22_chunk(X_chunk: np.ndarray) -> np.ndarray:
    transformer = Catch22(catch24=False)
    return transformer.fit_transform(X_chunk).to_numpy()


def extract_catch22(X: np.ndarray, label: str = "", n_jobs: int = -1, chunk_size: int = 500) -> np.ndarray:
    n_samples, n_timepoints, n_channels = X.shape

    if not np.isfinite(X).all():
        n_bad = (~np.isfinite(X)).sum()
        print(f"    ⚠️  {n_bad} non-finite values in {label} — replacing with per-channel median")

        X = X.copy()
        for c in range(n_channels):
            col = X[:, :, c]
            col[~np.isfinite(col)] = np.nanmedian(col)
            X[:, :, c] = col

    # sktime expects: (samples, channels, timesteps)
    X_sktime = X.transpose(0, 2, 1).astype(np.float64)

    chunks = [
        X_sktime[i:i + chunk_size]
        for i in range(0, n_samples, chunk_size)
    ]

    print(f"    {label} — {n_samples} samples in {len(chunks)} chunks …", flush=True)

    results = Parallel(n_jobs=n_jobs)(
        delayed(_catch22_chunk)(chunk)
        for chunk in chunks
    )

    X_feat = np.vstack(results)

    expected_cols = 22 * n_channels
    assert X_feat.shape == (n_samples, expected_cols), (
        f"Unexpected catch22 output shape: {X_feat.shape} "
        f"(expected ({n_samples}, {expected_cols}))"
    )

    return X_feat


# ── Sanity check ───────────────────────────────────────────────
print("Pre-extraction value range check:")
print(f"{'Norm':<14} {'Finite?':<10} {'Min':>15} {'Max':>15} {'NaN count':>12}")
print("─" * 70)

for norm_name, d in datasets.items():
    X = d["X_train"]
    print(
        f"{norm_name:<14} "
        f"{str(np.isfinite(X).all()):<10} "
        f"{X.min():>15.4f} "
        f"{X.max():>15.4f} "
        f"{(~np.isfinite(X)).sum():>12}"
    )


# ── Extraction loop: no saving, memory only ────────────────────
print("\nExtracting Catch22 features …")

datasets_svm = {}

for norm_name, d in datasets.items():
    print(f"\n[{norm_name}] extracting train …", flush=True)
    X_tr = extract_catch22(
        d["X_train"],
        label=f"{norm_name}/train",
        n_jobs=-1,
        chunk_size=200
    )

    print(f"[{norm_name}] extracting val …", flush=True)
    X_va = extract_catch22(
        d["X_val"],
        label=f"{norm_name}/val",
        n_jobs=-1,
        chunk_size=200
    )

    print(f"[{norm_name}] extracting test …", flush=True)
    X_te = extract_catch22(
        d["X_test"],
        label=f"{norm_name}/test",
        n_jobs=-1,
        chunk_size=200
    )

    datasets_svm[norm_name] = {
        "X_train": X_tr,
        "y_train": d["y_train"],
        "X_val":   X_va,
        "y_val":   d["y_val"],
        "X_test":  X_te,
        "y_test":  d["y_test"],
    }

    print(
        f"[{norm_name}] ✓  "
        f"train {X_tr.shape}  "
        f"val {X_va.shape}  "
        f"test {X_te.shape}"
    )


print("\n✅ Catch22 extraction complete.")
print(f"    Shape : 22 features × 10 channels = 220 cols per sample")
print(f"    Norms : {list(datasets_svm.keys())}")


# ══════════════════════════════════════════════════════════════
# IMPUTE — catch22 features
# Fit imputer on train only, apply to val/test
# ══════════════════════════════════════════════════════════════

for norm_name in list(datasets_svm.keys()):
    d = datasets_svm[norm_name]

    X_tr, y_tr = d["X_train"].copy(), d["y_train"]
    X_va, y_va = d["X_val"].copy(),   d["y_val"]
    X_te, y_te = d["X_test"].copy(),  d["y_test"]

    n_bad_tr = (~np.isfinite(X_tr)).sum()
    n_bad_va = (~np.isfinite(X_va)).sum()
    n_bad_te = (~np.isfinite(X_te)).sum()

    if n_bad_tr > 0 or n_bad_va > 0 or n_bad_te > 0:
        print(
            f"  ⚠️  [{norm_name}] non-finite values — "
            f"train: {n_bad_tr}  val: {n_bad_va}  test: {n_bad_te}  → imputing"
        )

        X_tr = np.where(np.isfinite(X_tr), X_tr, np.nan)
        X_va = np.where(np.isfinite(X_va), X_va, np.nan)
        X_te = np.where(np.isfinite(X_te), X_te, np.nan)

        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            imp = SimpleImputer(strategy="median")

            X_tr = imp.fit_transform(X_tr)
            X_va = imp.transform(X_va)
            X_te = imp.transform(X_te)

        X_tr = np.nan_to_num(X_tr, nan=0.0, posinf=0.0, neginf=0.0)
        X_va = np.nan_to_num(X_va, nan=0.0, posinf=0.0, neginf=0.0)
        X_te = np.nan_to_num(X_te, nan=0.0, posinf=0.0, neginf=0.0)

        print(f"  ✓  [{norm_name}] imputed")
    else:
        print(f"  ✓  [{norm_name}] no NaN or Inf — no imputation needed")

    datasets_svm[norm_name] = {
        "X_train": X_tr.astype(np.float32),
        "y_train": y_tr,
        "X_val":   X_va.astype(np.float32),
        "y_val":   y_va,
        "X_test":  X_te.astype(np.float32),
        "y_test":  y_te,
    }


print(f"\n✅ datasets_svm ready: {list(datasets_svm.keys())}")

Pre-extraction value range check:
Norm           Finite?                Min             Max    NaN count
──────────────────────────────────────────────────────────────────────
zscore_all     True               -6.5136        239.2680            0
minmax_all     True               -0.0000          1.0000            0
log_all        True             -137.8063         18.1921            0
hybrid         True               -6.5136         27.8326            0

Extracting Catch22 features …

[zscore_all] extracting train …
    zscore_all/train — 12455 samples in 63 chunks …
[zscore_all] extracting val …
    zscore_all/val — 1780 samples in 9 chunks …
[zscore_all] extracting test …
    zscore_all/test — 3559 samples in 18 chunks …
[zscore_all] ✓  train (12455, 220)  val (1780, 220)  test (3559, 220)

[minmax_all] extracting train …
    minmax_all/train — 12455 samples in 63 chunks …
[minmax_all] extracting val …
    minmax_all/val — 1780 samples in 9 chunks …
[minmax_all] extracting test …
 

## Classification

### Helper Functions 

In [4]:
# ══════════════════════════════════════════════════════════════
# HELPER FUNCTIONS
# ══════════════════════════════════════════════════════════════
import numpy as np
from sklearn.metrics import confusion_matrix
from sklearn.svm import SVC
import time
import os

RESULTS_DIR = "./results"
os.makedirs(RESULTS_DIR, exist_ok=True)


def compute_metrics(y_true, y_pred):
    cm             = confusion_matrix(y_true, y_pred)
    TN, FP, FN, TP = cm.ravel()

    recall    = TP / (TP + FN) if (TP + FN) > 0 else 0.0
    precision = TP / (TP + FP) if (TP + FP) > 0 else 0.0
    f1        = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
    accuracy  = (TP + TN) / (TP + TN + FP + FN)

    tss  = recall - FP / (FP + TN) if (FP + TN) > 0 else 0.0

    expected = ((TP + FN) * (TP + FP) + (TN + FP) * (TN + FN)) / (TP + TN + FP + FN) ** 2
    hss1     = (accuracy - expected) / (1 - expected) if (1 - expected) > 0 else 0.0

    denom = ((TP + FN) * (FN + TN) + (TP + FP) * (FP + TN))
    hss2  = 2 * (TP * TN - FP * FN) / denom if denom > 0 else 0.0

    hits_random = (TP + FP) * (TP + FN) / (TP + TN + FP + FN)
    gss = (TP - hits_random) / (TP + FP + FN - hits_random) if (TP + FP + FN - hits_random) > 0 else 0.0

    return {
        'TP': int(TP), 'TN': int(TN), 'FP': int(FP), 'FN': int(FN),
        'tss': tss, 'hss1': hss1, 'hss2': hss2, 'gss': gss,
        'recall': recall, 'f1': f1, 'accuracy': accuracy
    }


def train_and_evaluate(model, X_train, y_train, X_test, y_test):
    t0         = time.time()
    model.fit(X_train, y_train)
    train_time = time.time() - t0

    t0         = time.time()
    y_pred     = model.predict(X_test)
    infer_time = time.time() - t0

    metrics = compute_metrics(y_test, y_pred)
    metrics['train_time'] = train_time
    metrics['infer_time'] = infer_time
    return metrics


def save_results(metrics_list, filepath):
    with open(filepath, 'w') as f:
        for m in metrics_list:
            line = (f"{m['TP']},{m['TN']},{m['FP']},{m['FN']},"
                    f"{m['tss']:.6f},{m['hss1']:.6f},{m['hss2']:.6f},{m['gss']:.6f},"
                    f"{m['recall']:.6f},{m['f1']:.6f},{m['accuracy']:.6f},"
                    f"{m['train_time']:.4f},{m['infer_time']:.4f}")
            f.write(line + "\n")


def print_results(metrics_list, title):
    keys = ['tss', 'hss1', 'hss2', 'gss', 'recall', 'f1', 'accuracy', 'train_time', 'infer_time']
    print(f"\n{'─'*55}")
    print(f"  {title}")
    print(f"{'─'*55}")
    for i, m in enumerate(metrics_list):
        print(f"  Run {i+1}: TP={m['TP']}  TN={m['TN']}  FP={m['FP']}  FN={m['FN']}")
        print(f"         TSS={m['tss']:.4f}  HSS1={m['hss1']:.4f}  HSS2={m['hss2']:.4f}  GSS={m['gss']:.4f}")
        print(f"         Recall={m['recall']:.4f}  F1={m['f1']:.4f}  Acc={m['accuracy']:.4f}")
        print(f"         Train={m['train_time']:.2f}s  Infer={m['infer_time']:.4f}s")
        print()
    print(f"  ── Average of {len(metrics_list)} runs ──")
    for k in keys:
        avg = np.mean([m[k] for m in metrics_list])
        print(f"  {k:<12} : {avg:.4f}")
    print(f"{'─'*55}")

### SVM

In [5]:
import os
import time
import copy
import warnings
import numpy as np

from sklearn.svm import SVC, LinearSVC
from sklearn.exceptions import ConvergenceWarning

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=ConvergenceWarning)

# ══════════════════════════════════════════════════════════════
# DATASET VARIANTS
# ══════════════════════════════════════════════════════════════

DATASET_NAMES = {
    "no_norm":    "No normalization",
    "zscore_all": "Z-score all features",
    "minmax_all": "Min-Max all features",
    "log_all":    "Log all features",
    "hybrid":     "Hybrid normalization",
}

# ══════════════════════════════════════════════════════════════
# SVM HYPERPARAMETER GRID
# Tune only on hybrid validation set
# ══════════════════════════════════════════════════════════════



SVM_GRID = [

    {"model": LinearSVC, "params": {"C": 0.01, "class_weight": "balanced", "max_iter": 10000}},

    {"model": LinearSVC, "params": {"C": 0.05, "class_weight": "balanced", "max_iter": 10000}},

    {"model": LinearSVC, "params": {"C": 0.1,  "class_weight": "balanced", "max_iter": 10000}},

    {"model": LinearSVC, "params": {"C": 0.5,  "class_weight": "balanced", "max_iter": 10000}},

    {"model": LinearSVC, "params": {"C": 1.0,  "class_weight": "balanced", "max_iter": 10000}},

    {"model": LinearSVC, "params": {"C": 3.0,  "class_weight": "balanced", "max_iter": 10000}},

    {"model": SVC, "params": {"kernel": "rbf", "C": 0.01,  "gamma": "scale", "class_weight": "balanced", "cache_size": 2000}},
    
    {"model": SVC, "params": {"kernel": "rbf", "C": 0.1,  "gamma": "scale", "class_weight": "balanced", "cache_size": 2000}},

    {"model": SVC, "params": {"kernel": "rbf", "C": 1.0,  "gamma": "scale", "class_weight": "balanced", "cache_size": 2000}},

    {"model": SVC, "params": {"kernel": "rbf", "C": 10.0, "gamma": "scale", "class_weight": "balanced", "cache_size": 2000}},

]


def combo_label(combo):
    ModelClass = combo["model"]
    params = combo["params"]

    cw = "balanced" if params.get("class_weight") == "balanced" else "none"

    if ModelClass == LinearSVC:
        return f"LinearSVC | C={params['C']} | cw={cw}"
    else:
        return f"RBF-SVC   | C={params['C']} | gamma={params['gamma']} | cw={cw}"


# ══════════════════════════════════════════════════════════════
# STEP 1: SEARCH BEST HYPERPARAMETERS ON HYBRID VALIDATION SET ONLY
# ══════════════════════════════════════════════════════════════

print("═" * 70)
print("SVM hyperparameter search on HYBRID validation set only")
print("═" * 70)

d_hybrid = datasets_svm["hybrid"]

best_tss = -np.inf
best_model_class = None
best_params = None
best_val_metrics = None

for combo in SVM_GRID:
    ModelClass = combo["model"]
    params = copy.deepcopy(combo["params"])

    model = ModelClass(**params)

    t0 = time.time()
    model.fit(d_hybrid["X_train"], d_hybrid["y_train"])
    y_val_pred = model.predict(d_hybrid["X_val"])
    elapsed = time.time() - t0

    val_metrics = compute_metrics(d_hybrid["y_val"], y_val_pred)

    print(
        f"{combo_label(combo):<55} "
        f"Val TSS={val_metrics['tss']:.4f} | "
        f"F1={val_metrics['f1']:.4f} | "
        f"{elapsed:.1f}s"
    )

    if val_metrics["tss"] > best_tss:
        best_tss = val_metrics["tss"]
        best_model_class = ModelClass
        best_params = params
        best_val_metrics = val_metrics

print("\nBest hyperparameters selected from hybrid validation set:")
print(f"  Model   : {best_model_class.__name__}")
print(f"  Params  : {best_params}")
print(f"  Val TSS : {best_tss:.4f}")
print(f"  Val F1  : {best_val_metrics['f1']:.4f}")


# ══════════════════════════════════════════════════════════════
# STEP 2: RUN FINAL EXPERIMENTS ON ALL NORMALIZATION VARIANTS
# using the same best hyperparameters selected from hybrid
# ══════════════════════════════════════════════════════════════

print("\n" + "═" * 70)
print("Final SVM experiments on all normalization variants")
print("Using best hyperparameters selected from hybrid validation")
print("═" * 70)

for norm_key, norm_label in DATASET_NAMES.items():

    if norm_key not in datasets_svm:
        print(f"\n[{norm_key}] not found — skipping")
        continue

    d = datasets_svm[norm_key]

    print("\n" + "─" * 70)
    print(f"Dataset: {norm_label}")
    print(f"Model  : {best_model_class.__name__}")
    print(f"Params : {best_params}")
    print("─" * 70)

    metrics_list = []

    for run in range(2):
        model = best_model_class(**copy.deepcopy(best_params))

        metrics = train_and_evaluate(
            model,
            d["X_train"], d["y_train"],
            d["X_test"],  d["y_test"]
        )

        metrics_list.append(metrics)

        print(
    f"Run {run + 1}: "
    f"TSS={metrics['tss']:.4f} | "
    f"F1={metrics['f1']:.4f} | "
    f"Recall={metrics['recall']:.4f} | "
    f"Train={metrics['train_time']:.2f}s"
)

    filepath = os.path.join(RESULTS_DIR, f"svm_{norm_key}.txt")
    save_results(metrics_list, filepath)
    print_results(metrics_list, f"SVM | {norm_label}")


print("\nAll SVM experiments complete.")
print(f"Results saved to: {os.path.abspath(RESULTS_DIR)}/")
print(f"Best model used for all variants: {best_model_class.__name__}")
print(f"Best params used for all variants: {best_params}")

══════════════════════════════════════════════════════════════════════
SVM hyperparameter search on HYBRID validation set only
══════════════════════════════════════════════════════════════════════
LinearSVC | C=0.01 | cw=balanced                        Val TSS=0.4485 | F1=0.0582 | 27.3s
LinearSVC | C=0.05 | cw=balanced                        Val TSS=0.4175 | F1=0.0610 | 98.6s
LinearSVC | C=0.1 | cw=balanced                         Val TSS=0.3587 | F1=0.0550 | 241.7s
LinearSVC | C=0.5 | cw=balanced                         Val TSS=0.2840 | F1=0.0452 | 269.1s
LinearSVC | C=1.0 | cw=balanced                         Val TSS=0.2715 | F1=0.0426 | 339.2s
LinearSVC | C=3.0 | cw=balanced                         Val TSS=0.3144 | F1=0.0444 | 392.6s
RBF-SVC   | C=0.01 | gamma=scale | cw=balanced          Val TSS=0.3977 | F1=0.0551 | 16.3s
RBF-SVC   | C=0.1 | gamma=scale | cw=balanced           Val TSS=0.4604 | F1=0.0616 | 11.2s
RBF-SVC   | C=1.0 | gamma=scale | cw=balanced           Val TSS=0.3197

### GRU 

In [7]:
# ══════════════════════════════════════════════════════════════
# PART 1 — GRU MODEL, GRID, AND HELPER FUNCTIONS
# ══════════════════════════════════════════════════════════════

import numpy as np
import torch
import torch.nn as nn
import time
import os
import copy
import warnings

warnings.filterwarnings("ignore")

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Using device: {device}")


# ── GRU model ─────────────────────────────────────────────────
class GRUModel(nn.Module):
    def __init__(self, input_size, hidden_size=128, num_layers=1, dropout=0.3):
        super(GRUModel, self).__init__()

        self.gru = nn.GRU(
            input_size,
            hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0
        )

        self.bn = nn.BatchNorm1d(hidden_size)
        self.dropout = nn.Dropout(dropout)
        self.fc1 = nn.Linear(hidden_size, 32)
        self.relu = nn.ReLU()
        self.dropout2 = nn.Dropout(0.2)
        self.fc2 = nn.Linear(32, 1)

    def forward(self, x):
        out, _ = self.gru(x)
        out = out[:, -1, :]          # use last hidden representation
        out = self.bn(out)
        out = self.dropout(out)
        out = self.relu(self.fc1(out))
        out = self.dropout2(out)
        return self.fc2(out).squeeze()


# ── GRU hyperparameter grid ───────────────────────────────────
GRU_GRID = [

    # Small models

    {"units": 32,  "dropout": 0.1, "layers": 1, "lr": 0.001,  "batch_size": 64},

    {"units": 32,  "dropout": 0.3, "layers": 1, "lr": 0.001,  "batch_size": 64},

    # Medium models

    {"units": 64,  "dropout": 0.1, "layers": 1, "lr": 0.001,  "batch_size": 64},

    {"units": 64,  "dropout": 0.3, "layers": 1, "lr": 0.001,  "batch_size": 64},


    # Larger single-layer models

    {"units": 128, "dropout": 0.1, "layers": 1, "lr": 0.001,  "batch_size": 32},

    {"units": 128, "dropout": 0.3, "layers": 1, "lr": 0.001,  "batch_size": 32},

    # Two-layer models

    {"units": 64,  "dropout": 0.1, "layers": 2, "lr": 0.001,  "batch_size": 64},

    {"units": 128, "dropout": 0.3, "layers": 2, "lr": 0.001,  "batch_size": 32},

    # Lower learning rate variants

    {"units": 64,  "dropout": 0.3, "layers": 2, "lr": 0.0005, "batch_size": 64},

    {"units": 128, "dropout": 0.3, "layers": 2, "lr": 0.0005, "batch_size": 32},

]


def gru_label(p):
    return (
        f"units={p['units']}  "
        f"layers={p['layers']}  "
        f"drop={p['dropout']}  "
        f"lr={p['lr']}  "
        f"bs={p['batch_size']}"
    )


# ── Train GRU and evaluate on a provided test split ────────────
def train_and_evaluate_gru(
    params,
    X_train, y_train,
    X_val, y_val,
    X_test, y_test,
    class_ratio,
    max_epochs=50,
    patience=5
):
    # Convert numpy arrays to tensors
    X_tr = torch.tensor(X_train, dtype=torch.float32).to(device)
    y_tr = torch.tensor(y_train, dtype=torch.float32).to(device)

    X_va = torch.tensor(X_val, dtype=torch.float32).to(device)
    y_va = torch.tensor(y_val, dtype=torch.float32).to(device)

    X_te = torch.tensor(X_test, dtype=torch.float32).to(device)

    # Build model
    input_size = X_train.shape[2]

    model = GRUModel(
        input_size=input_size,
        hidden_size=params["units"],
        num_layers=params["layers"],
        dropout=params["dropout"]
    ).to(device)

    # Weighted BCE loss for class imbalance
    pos_weight = torch.tensor([float(class_ratio)], dtype=torch.float32).to(device)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

    optimizer = torch.optim.Adam(model.parameters(), lr=params["lr"])

    batch_size = params["batch_size"]
    n_samples = X_tr.shape[0]

    best_val_loss = float("inf")
    best_state = None
    patience_count = 0

    t0 = time.time()

    for epoch in range(max_epochs):
        model.train()

        # Shuffle training samples each epoch
        perm = torch.randperm(n_samples, device=device)

        for i in range(0, n_samples, batch_size):
            idx = perm[i:i + batch_size]

            xb = X_tr[idx]
            yb = y_tr[idx]

            optimizer.zero_grad()
            logits = model(xb)
            loss = criterion(logits, yb)
            loss.backward()
            optimizer.step()

        # Validation loss for early stopping
        model.eval()
        with torch.no_grad():
            val_logits = model(X_va)
            val_loss = criterion(val_logits, y_va).item()

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            patience_count = 0
            best_state = copy.deepcopy(model.state_dict())
        else:
            patience_count += 1
            if patience_count >= patience:
                break

    train_time = time.time() - t0

    # Load best validation-loss model
    if best_state is not None:
        model.load_state_dict(best_state)

    # Test prediction
    model.eval()
    t0 = time.time()

    with torch.no_grad():
        y_prob = torch.sigmoid(model(X_te)).cpu().numpy()

    y_pred = (y_prob >= 0.5).astype(int)
    infer_time = time.time() - t0

    metrics = compute_metrics(y_test.astype(int), y_pred)
    metrics["train_time"] = train_time
    metrics["infer_time"] = infer_time

    return metrics, model

Using device: mps


### Run GRU Experiments

In [6]:
# ══════════════════════════════════════════════════════════════
# PART 2 — RUN GRU EXPERIMENTS
# Tune on hybrid validation set only, then test all variants
# ══════════════════════════════════════════════════════════════

DATASET_NAMES = {
    "no_norm":    "No normalization",
    "zscore_all": "Z-score all features",
    "minmax_all": "Min-Max all features",
    "log_all":    "Log all features",
    "hybrid":     "Hybrid normalization",
}

print(f"{'═' * 60}")
print("  Classifier : GRU")
print(f"{'═' * 60}")

# ── Step 1: Hyperparameter search on hybrid validation set only ──────────────
print(f"\nSearching {len(GRU_GRID)} GRU combinations on hybrid validation set only...\n")

# Use the loaded MVTS sequence datasets for GRU
# Shape should be: (samples, 288, 10)
datasets_gru = datasets

print("GRU datasets:", list(datasets_gru.keys()))
for k, d in datasets_gru.items():
    print(k, d["X_train"].shape)

d_hybrid = datasets_gru["hybrid"]

class_ratio = int(
    (d_hybrid["y_train"] == 0).sum() /
    (d_hybrid["y_train"] == 1).sum()
)

print(f"Hybrid train class ratio 0:1 = {class_ratio}:1\n")

best_params = None
best_tss = -999
best_val_metrics = None

for i, params in enumerate(GRU_GRID):
    print(f"[{i + 1}/{len(GRU_GRID)}] {gru_label(params)}", flush=True)

    # Here validation set is used as the evaluation split
    metrics, _ = train_and_evaluate_gru(
        params,
        d_hybrid["X_train"], d_hybrid["y_train"],
        d_hybrid["X_val"],   d_hybrid["y_val"],
        d_hybrid["X_val"],   d_hybrid["y_val"],
        class_ratio=class_ratio
    )

    print(
        f"    Val TSS={metrics['tss']:.4f}  "
        f"F1={metrics['f1']:.4f}  "
        f"Recall={metrics['recall']:.4f}  "
        f"Train={metrics['train_time']:.1f}s\n"
    )

    if metrics["tss"] > best_tss:
        best_tss = metrics["tss"]
        best_params = copy.deepcopy(params)
        best_val_metrics = metrics

print("Best GRU hyperparameters selected from hybrid validation set:")
print(f"  Params  : {gru_label(best_params)}")
print(f"  Val TSS : {best_tss:.4f}")
print(f"  Val F1  : {best_val_metrics['f1']:.4f}")


# ── Step 2: Run final experiments on all normalization variants ─────────────
print("\nRunning final GRU experiments on all normalization variants...")
print("Using the same best hyperparameters selected from hybrid.\n")

for norm_key, norm_label in DATASET_NAMES.items():

    if norm_key not in datasets_gru:
        print(f"[{norm_key}] not found — skipping")
        continue

    d = datasets_gru[norm_key]

    # Compute class ratio separately for each training set
    cr = int(
        (d["y_train"] == 0).sum() /
        (d["y_train"] == 1).sum()
    )

    print("\n" + "─" * 70)
    print(f"Dataset : {norm_label}")
    print(f"Params  : {gru_label(best_params)}")
    print(f"Ratio   : {cr}:1")
    print("─" * 70)

    metrics_list = []

    for run in range(2):
        print(f"Run {run + 1}...", flush=True)

        metrics, _ = train_and_evaluate_gru(
            best_params,
            d["X_train"], d["y_train"],
            d["X_val"],   d["y_val"],
            d["X_test"],  d["y_test"],
            class_ratio=cr
        )

        metrics_list.append(metrics)

        print(
            f"Run {run + 1} — "
            f"TSS={metrics['tss']:.4f}  "
            f"F1={metrics['f1']:.4f}  "
            f"Recall={metrics['recall']:.4f}  "
            f"Train={metrics['train_time']:.1f}s"
        )

    filepath = os.path.join(RESULTS_DIR, f"gru_{norm_key}.txt")
    save_results(metrics_list, filepath)
    print_results(metrics_list, f"GRU | {norm_label}")


print("\nAll GRU experiments complete.")
print(f"Results saved to: {os.path.abspath(RESULTS_DIR)}/")
print(f"Best GRU params used for all variants: {gru_label(best_params)}")

════════════════════════════════════════════════════════════
  Classifier : GRU
════════════════════════════════════════════════════════════

Searching 10 GRU combinations on hybrid validation set only...

GRU datasets: ['zscore_all', 'minmax_all', 'log_all', 'hybrid']
zscore_all (12455, 288, 10)
minmax_all (12455, 288, 10)
log_all (12455, 288, 10)
hybrid (12455, 288, 10)
Hybrid train class ratio 0:1 = 104:1

[1/10] units=32  layers=1  drop=0.1  lr=0.001  bs=64
    Val TSS=0.5448  F1=0.0767  Recall=0.7059  Train=107.5s

[2/10] units=32  layers=1  drop=0.3  lr=0.001  bs=64
    Val TSS=0.6488  F1=0.0826  Recall=0.8235  Train=94.3s

[3/10] units=64  layers=1  drop=0.1  lr=0.001  bs=64
    Val TSS=0.6074  F1=0.0680  Recall=0.8235  Train=119.4s

[4/10] units=64  layers=1  drop=0.3  lr=0.001  bs=64
    Val TSS=0.6233  F1=0.0729  Recall=0.8235  Train=107.2s

[5/10] units=128  layers=1  drop=0.1  lr=0.001  bs=32
    Val TSS=0.5842  F1=0.0618  Recall=0.8235  Train=314.9s

[6/10] units=128  laye

### PatchTST

In [18]:
# ══════════════════════════════════════════════════════════════
# PART 1 — HUGGING FACE PATCHTST MODEL + TRAINING FUNCTION
# ══════════════════════════════════════════════════════════════

import os
import time
import copy
import warnings
import numpy as np

import torch
import torch.nn as nn
from transformers import PatchTSTConfig, PatchTSTForClassification

warnings.filterwarnings("ignore")

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Using device: {device}")


# ── PatchTST hyperparameter grid ──────────────────────────────
PATCHTST_GRID = [
    # Best current region from your results
    {
        "patch_length": 6,
        "patch_stride": 6,
        "d_model": 16,
        "num_attention_heads": 4,
        "num_hidden_layers": 1,
        "ffn_dim": 64,
        "dropout": 0.2,
        "lr": 1e-3,
        "batch_size": 128,
    },
    {
        "patch_length": 6,
        "patch_stride": 6,
        "d_model": 32,
        "num_attention_heads": 4,
        "num_hidden_layers": 1,
        "ffn_dim": 128,
        "dropout": 0.1,
        "lr": 1e-3,
        "batch_size": 128,
    },
    {
        "patch_length": 6,
        "patch_stride": 6,
        "d_model": 32,
        "num_attention_heads": 4,
        "num_hidden_layers": 1,
        "ffn_dim": 128,
        "dropout": 0.2,
        "lr": 1e-3,
        "batch_size": 128,
    },
    {
        "patch_length": 6,
        "patch_stride": 6,
        "d_model": 64,
        "num_attention_heads": 4,
        "num_hidden_layers": 1,
        "ffn_dim": 256,
        "dropout": 0.2,
        "lr": 1e-3,
        "batch_size": 128,
    },

    # Patch=12 region: better F1 in your earlier run
    {
        "patch_length": 12,
        "patch_stride": 12,
        "d_model": 16,
        "num_attention_heads": 4,
        "num_hidden_layers": 1,
        "ffn_dim": 64,
        "dropout": 0.2,
        "lr": 1e-3,
        "batch_size": 128,
    },
    {
        "patch_length": 12,
        "patch_stride": 12,
        "d_model": 32,
        "num_attention_heads": 4,
        "num_hidden_layers": 1,
        "ffn_dim": 128,
        "dropout": 0.2,
        "lr": 1e-3,
        "batch_size": 128,
    },
    {
        "patch_length": 12,
        "patch_stride": 12,
        "d_model": 64,
        "num_attention_heads": 4,
        "num_hidden_layers": 1,
        "ffn_dim": 256,
        "dropout": 0.2,
        "lr": 1e-3,
        "batch_size": 128,
    },

    # Coarser patch baseline
    {
        "patch_length": 24,
        "patch_stride": 24,
        "d_model": 64,
        "num_attention_heads": 4,
        "num_hidden_layers": 1,
        "ffn_dim": 256,
        "dropout": 0.2,
        "lr": 1e-3,
        "batch_size": 128,
    },
]


def patchtst_label(p):
    return (
        f"patch={p['patch_length']}  "
        f"stride={p['patch_stride']}  "
        f"d={p['d_model']}  "
        f"heads={p['num_attention_heads']}  "
        f"layers={p['num_hidden_layers']}  "
        f"drop={p['dropout']}  "
        f"lr={p['lr']}  "
        f"bs={p['batch_size']}"
    )


def build_patchtst_model(params, seq_len=288, n_channels=10):
    # num_targets=2 because this is binary classification: NSEP vs SEP
    config = PatchTSTConfig(
        num_input_channels=n_channels,
        context_length=seq_len,
        num_targets=2,

        patch_length=params["patch_length"],
        patch_stride=params["patch_stride"],

        d_model=params["d_model"],
        num_attention_heads=params["num_attention_heads"],
        num_hidden_layers=params["num_hidden_layers"],
        ffn_dim=params["ffn_dim"],

        attention_dropout=params["dropout"],
        positional_dropout=params["dropout"],
        ff_dropout=params["dropout"],
        head_dropout=params["dropout"],

        pooling_type="mean",
        use_cls_token=True,
        norm_type="layernorm",
        channel_attention=True,
        problem_type="single_label_classification",
        scaling=None,
    )

    model = PatchTSTForClassification(config)
    return model


def train_and_evaluate_patchtst(
    params,
    X_train, y_train,
    X_val, y_val,
    X_test, y_test,
    class_ratio,
    max_epochs=30,
    patience=3
):
    # X shape must be: (samples, timesteps, features)
    X_tr = torch.tensor(X_train, dtype=torch.float32).to(device)
    y_tr = torch.tensor(y_train, dtype=torch.long).to(device)

    X_va = torch.tensor(X_val, dtype=torch.float32).to(device)
    y_va = torch.tensor(y_val, dtype=torch.long).to(device)

    X_te = torch.tensor(X_test, dtype=torch.float32).to(device)

    seq_len = X_train.shape[1]
    n_channels = X_train.shape[2]

    model = build_patchtst_model(
        params,
        seq_len=seq_len,
        n_channels=n_channels
    ).to(device)

    # Class-weighted CE loss for imbalanced SEP classification
    class_weights = torch.tensor(
        [1.0, float(class_ratio)],
        dtype=torch.float32
    ).to(device)

    criterion = nn.CrossEntropyLoss(weight=class_weights)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=params["lr"],
        weight_decay=1e-4
    )

    batch_size = params["batch_size"]
    n_samples = X_tr.shape[0]

    best_val_loss = float("inf")
    best_state = None
    patience_count = 0

    t0 = time.time()

    for epoch in range(max_epochs):
        model.train()

        perm = torch.randperm(n_samples, device=device)
        epoch_loss = 0.0

        for i in range(0, n_samples, batch_size):
            idx = perm[i:i + batch_size]

            xb = X_tr[idx]
            yb = y_tr[idx]

            optimizer.zero_grad()

            outputs = model(past_values=xb)
            logits = outputs.prediction_logits

            loss = criterion(logits, yb)
            loss.backward()

            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

            optimizer.step()

            epoch_loss += loss.item()

        # Validation loss for early stopping
        model.eval()
        with torch.no_grad():
            val_outputs = model(past_values=X_va)
            val_logits = val_outputs.prediction_logits
            val_loss = criterion(val_logits, y_va).item()

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = copy.deepcopy(model.state_dict())
            patience_count = 0
        else:
            patience_count += 1
            if patience_count >= patience:
                break

    train_time = time.time() - t0

    if best_state is not None:
        model.load_state_dict(best_state)

    # Test prediction
    model.eval()
    t0 = time.time()

    with torch.no_grad():
        test_outputs = model(past_values=X_te)
        test_logits = test_outputs.prediction_logits
        y_pred = torch.argmax(test_logits, dim=1).cpu().numpy()

    infer_time = time.time() - t0

    metrics = compute_metrics(y_test.astype(int), y_pred.astype(int))
    metrics["train_time"] = train_time
    metrics["infer_time"] = infer_time

    return metrics, model

Using device: mps


In [19]:
# ══════════════════════════════════════════════════════════════
# PART 2 — PATCHTST EXPERIMENTS
# Tune on hybrid validation only, then run all variants
# ══════════════════════════════════════════════════════════════

# Use sequence datasets, not Catch22 features
datasets_patchtst = datasets

DATASET_NAMES = {
    "zscore_all": "Z-score all features",
    "minmax_all": "Min-Max all features",
    "log_all":    "Log all features",
    "hybrid":     "Hybrid normalization",
}

print(f"{'═' * 60}")
print("  Classifier : Hugging Face PatchTST")
print(f"{'═' * 60}")

# ── Step 1: Hyperparameter search on hybrid validation set only ──────────────
print(f"\nSearching {len(PATCHTST_GRID)} PatchTST combinations on hybrid validation set only...\n")

d_hybrid = datasets_patchtst["hybrid"]

class_ratio = int(
    (d_hybrid["y_train"] == 0).sum() /
    (d_hybrid["y_train"] == 1).sum()
)

print(f"Hybrid train class ratio 0:1 = {class_ratio}:1\n")

best_params = None
best_tss = -999
best_val_metrics = None

for i, params in enumerate(PATCHTST_GRID):
    print(f"[{i + 1}/{len(PATCHTST_GRID)}] {patchtst_label(params)}", flush=True)

    # During search, evaluate on hybrid validation set
    metrics, _ = train_and_evaluate_patchtst(
        params,
        d_hybrid["X_train"], d_hybrid["y_train"],
        d_hybrid["X_val"],   d_hybrid["y_val"],
        d_hybrid["X_val"],   d_hybrid["y_val"],
        class_ratio=class_ratio,
        max_epochs=30,
        patience=3
    )

    print(
        f"    Val TSS={metrics['tss']:.4f}  "
        f"F1={metrics['f1']:.4f}  "
        f"Recall={metrics['recall']:.4f}  "
        f"Train={metrics['train_time']:.1f}s\n"
    )

    if metrics["tss"] > best_tss:
        best_tss = metrics["tss"]
        best_params = copy.deepcopy(params)
        best_val_metrics = metrics

print("Best PatchTST hyperparameters selected from hybrid validation set:")
print(f"  Params  : {patchtst_label(best_params)}")
print(f"  Val TSS : {best_tss:.4f}")
print(f"  Val F1  : {best_val_metrics['f1']:.4f}")


# ── Step 2: Run final experiments on all normalization variants ──────────────
print("\nRunning final PatchTST experiments on all normalization variants...")
print("Using the same best hyperparameters selected from hybrid.\n")

for norm_key, norm_label in DATASET_NAMES.items():

    if norm_key not in datasets_patchtst:
        print(f"[{norm_key}] not found — skipping")
        continue

    d = datasets_patchtst[norm_key]

    cr = int(
        (d["y_train"] == 0).sum() /
        (d["y_train"] == 1).sum()
    )

    print("\n" + "─" * 70)
    print(f"Dataset : {norm_label}")
    print(f"Params  : {patchtst_label(best_params)}")
    print(f"Ratio   : {cr}:1")
    print("─" * 70)

    metrics_list = []

    for run in range(2):
        print(f"Run {run + 1}...", flush=True)

        metrics, _ = train_and_evaluate_patchtst(
            best_params,
            d["X_train"], d["y_train"],
            d["X_val"],   d["y_val"],
            d["X_test"],  d["y_test"],
            class_ratio=cr,
            max_epochs=30,
            patience=3
        )

        metrics_list.append(metrics)

        print(
            f"Run {run + 1} — "
            f"TSS={metrics['tss']:.4f}  "
            f"F1={metrics['f1']:.4f}  "
            f"Recall={metrics['recall']:.4f}  "
            f"Train={metrics['train_time']:.1f}s"
        )

    filepath = os.path.join(RESULTS_DIR, f"patchtst_{norm_key}.txt")
    save_results(metrics_list, filepath)
    print_results(metrics_list, f"PatchTST | {norm_label}")


print("\nAll PatchTST experiments complete.")
print(f"Results saved to: {os.path.abspath(RESULTS_DIR)}/")
print(f"Best PatchTST params used for all variants: {patchtst_label(best_params)}")

════════════════════════════════════════════════════════════
  Classifier : Hugging Face PatchTST
════════════════════════════════════════════════════════════

Searching 8 PatchTST combinations on hybrid validation set only...

Hybrid train class ratio 0:1 = 104:1

[1/8] patch=6  stride=6  d=16  heads=4  layers=1  drop=0.2  lr=0.001  bs=128
    Val TSS=0.5749  F1=0.0923  Recall=0.7059  Train=46.3s

[2/8] patch=6  stride=6  d=32  heads=4  layers=1  drop=0.1  lr=0.001  bs=128
    Val TSS=0.5709  F1=0.0899  Recall=0.7059  Train=66.4s

[3/8] patch=6  stride=6  d=32  heads=4  layers=1  drop=0.2  lr=0.001  bs=128
    Val TSS=0.6354  F1=0.1008  Recall=0.7647  Train=70.3s

[4/8] patch=6  stride=6  d=64  heads=4  layers=1  drop=0.2  lr=0.001  bs=128
    Val TSS=0.5352  F1=0.0727  Recall=0.7059  Train=63.6s

[5/8] patch=12  stride=12  d=16  heads=4  layers=1  drop=0.2  lr=0.001  bs=128
    Val TSS=0.6127  F1=0.0872  Recall=0.7647  Train=23.9s

[6/8] patch=12  stride=12  d=32  heads=4  layers=1  

### InceptionTime

In [20]:
# ══════════════════════════════════════════════════════════════
# PART 1 — INCEPTIONTIME MODEL, GRID, AND TRAINING FUNCTION
# ══════════════════════════════════════════════════════════════

import os
import time
import copy
import warnings
import numpy as np

import torch
import torch.nn as nn

warnings.filterwarnings("ignore")

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Using device: {device}")


# ── Inception module ──────────────────────────────────────────
class InceptionModule(nn.Module):
    def __init__(self, in_channels, out_channels=32, kernel_sizes=(9, 19, 39), bottleneck_channels=32):
        super(InceptionModule, self).__init__()

        # 1x1 bottleneck to reduce channel dimension
        if in_channels > 1:
            self.bottleneck = nn.Conv1d(in_channels, bottleneck_channels, kernel_size=1, bias=False)
            conv_in_channels = bottleneck_channels
        else:
            self.bottleneck = nn.Identity()
            conv_in_channels = in_channels

        self.conv_list = nn.ModuleList([
            nn.Conv1d(
                conv_in_channels,
                out_channels,
                kernel_size=k,
                padding=k // 2,
                bias=False
            )
            for k in kernel_sizes
        ])

        self.maxpool_conv = nn.Sequential(
            nn.MaxPool1d(kernel_size=3, stride=1, padding=1),
            nn.Conv1d(in_channels, out_channels, kernel_size=1, bias=False)
        )

        total_out_channels = out_channels * (len(kernel_sizes) + 1)

        self.bn = nn.BatchNorm1d(total_out_channels)
        self.relu = nn.ReLU()

    def forward(self, x):
        x_bottleneck = self.bottleneck(x)

        conv_outputs = [
            conv(x_bottleneck)
            for conv in self.conv_list
        ]

        pool_output = self.maxpool_conv(x)

        out = torch.cat(conv_outputs + [pool_output], dim=1)
        out = self.bn(out)
        out = self.relu(out)

        return out


# ── Inception block with optional residual connection ─────────
class InceptionBlock(nn.Module):
    def __init__(self, in_channels, out_channels=32, depth=3, use_residual=True):
        super(InceptionBlock, self).__init__()

        self.use_residual = use_residual
        self.depth = depth

        modules = []
        current_channels = in_channels

        for _ in range(depth):
            module = InceptionModule(
                in_channels=current_channels,
                out_channels=out_channels
            )
            modules.append(module)

            # each InceptionModule outputs 4 branches × out_channels
            current_channels = out_channels * 4

        self.inception_modules = nn.ModuleList(modules)

        if use_residual:
            self.residual = nn.Sequential(
                nn.Conv1d(in_channels, current_channels, kernel_size=1, bias=False),
                nn.BatchNorm1d(current_channels)
            )
            self.relu = nn.ReLU()

    def forward(self, x):
        residual = x

        out = x
        for module in self.inception_modules:
            out = module(out)

        if self.use_residual:
            residual = self.residual(residual)
            out = self.relu(out + residual)

        return out


# ── InceptionTime classifier ──────────────────────────────────
class InceptionTimeClassifier(nn.Module):
    def __init__(self, input_channels, out_channels=32, depth=6, dropout=0.3):
        super(InceptionTimeClassifier, self).__init__()

        # Use two residual blocks if depth=6
        # Each block contains depth//2 Inception modules
        block_depth = max(1, depth // 2)

        self.block1 = InceptionBlock(
            in_channels=input_channels,
            out_channels=out_channels,
            depth=block_depth,
            use_residual=True
        )

        block1_channels = out_channels * 4

        self.block2 = InceptionBlock(
            in_channels=block1_channels,
            out_channels=out_channels,
            depth=block_depth,
            use_residual=True
        )

        final_channels = out_channels * 4

        self.gap = nn.AdaptiveAvgPool1d(1)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(final_channels, 1)

    def forward(self, x):
        # input x shape: (batch, timesteps, features)
        # Conv1d expects: (batch, features, timesteps)
        x = x.permute(0, 2, 1)

        x = self.block1(x)
        x = self.block2(x)

        x = self.gap(x).squeeze(-1)
        x = self.dropout(x)

        return self.fc(x).squeeze()


# ── InceptionTime hyperparameter grid ─────────────────────────
INCEPTION_GRID = [
    {"out_channels": 8, "depth": 4,  "dropout": 0.1, "lr": 1e-3,  "batch_size": 128},
    {"out_channels": 8, "depth": 4,  "dropout": 0.2, "lr": 1e-3,  "batch_size": 128},
    {"out_channels": 16, "depth": 4,  "dropout": 0.1, "lr": 1e-3,  "batch_size": 128},
    {"out_channels": 16, "depth": 4,  "dropout": 0.2, "lr": 1e-3,  "batch_size": 128},
    {"out_channels": 16, "depth": 6,  "dropout": 0.1, "lr": 1e-3,  "batch_size": 128},
    {"out_channels": 16, "depth": 6,  "dropout": 0.2, "lr": 1e-3,  "batch_size": 128},
    {"out_channels": 16, "depth": 6,  "dropout": 0.3, "lr": 1e-3,  "batch_size": 128},
    {"out_channels": 32, "depth": 6,  "dropout": 0.2, "lr": 1e-3,  "batch_size": 64},
]


def inception_label(p):
    return (
        f"channels={p['out_channels']}  "
        f"depth={p['depth']}  "
        f"drop={p['dropout']}  "
        f"lr={p['lr']}  "
        f"bs={p['batch_size']}"
    )


# ── Train and evaluate InceptionTime ──────────────────────────
def train_and_evaluate_inception(
    params,
    X_train, y_train,
    X_val, y_val,
    X_test, y_test,
    class_ratio,
    max_epochs=30,
    patience=3
):
    X_tr = torch.tensor(X_train, dtype=torch.float32).to(device)
    y_tr = torch.tensor(y_train, dtype=torch.float32).to(device)

    X_va = torch.tensor(X_val, dtype=torch.float32).to(device)
    y_va = torch.tensor(y_val, dtype=torch.float32).to(device)

    X_te = torch.tensor(X_test, dtype=torch.float32).to(device)

    input_channels = X_train.shape[2]

    model = InceptionTimeClassifier(
        input_channels=input_channels,
        out_channels=params["out_channels"],
        depth=params["depth"],
        dropout=params["dropout"]
    ).to(device)

    # Weighted BCE loss for SEP imbalance
    pos_weight = torch.tensor([float(class_ratio)], dtype=torch.float32).to(device)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=params["lr"],
        weight_decay=1e-4
    )

    batch_size = params["batch_size"]
    n_samples = X_tr.shape[0]

    best_val_loss = float("inf")
    best_state = None
    patience_count = 0

    t0 = time.time()

    for epoch in range(max_epochs):
        model.train()

        perm = torch.randperm(n_samples, device=device)

        for i in range(0, n_samples, batch_size):
            idx = perm[i:i + batch_size]

            xb = X_tr[idx]
            yb = y_tr[idx]

            optimizer.zero_grad()

            logits = model(xb)
            loss = criterion(logits, yb)

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

        # validation loss for early stopping
        model.eval()
        with torch.no_grad():
            val_logits = model(X_va)
            val_loss = criterion(val_logits, y_va).item()

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = copy.deepcopy(model.state_dict())
            patience_count = 0
        else:
            patience_count += 1
            if patience_count >= patience:
                break

    train_time = time.time() - t0

    if best_state is not None:
        model.load_state_dict(best_state)

    model.eval()
    t0 = time.time()

    with torch.no_grad():
        y_prob = torch.sigmoid(model(X_te)).cpu().numpy()

    y_pred = (y_prob >= 0.5).astype(int)
    infer_time = time.time() - t0

    metrics = compute_metrics(y_test.astype(int), y_pred.astype(int))
    metrics["train_time"] = train_time
    metrics["infer_time"] = infer_time

    return metrics, model

Using device: mps


In [21]:
# ══════════════════════════════════════════════════════════════
# PART 2 — INCEPTIONTIME EXPERIMENTS
# Tune on hybrid validation only, then run all variants
# ══════════════════════════════════════════════════════════════

datasets_inception = datasets

DATASET_NAMES = {
    "zscore_all": "Z-score all features",
    "minmax_all": "Min-Max all features",
    "log_all":    "Log all features",
    "hybrid":     "Hybrid normalization",
}

print(f"{'═' * 60}")
print("  Classifier : InceptionTime")
print(f"{'═' * 60}")

# ── Step 1: Hyperparameter search on hybrid validation set only ─────────────
print(f"\nSearching {len(INCEPTION_GRID)} InceptionTime combinations on hybrid validation set only...\n")

d_hybrid = datasets_inception["hybrid"]

class_ratio = int(
    (d_hybrid["y_train"] == 0).sum() /
    (d_hybrid["y_train"] == 1).sum()
)

print(f"Hybrid train class ratio 0:1 = {class_ratio}:1\n")

best_params = None
best_tss = -999
best_val_metrics = None

for i, params in enumerate(INCEPTION_GRID):
    print(f"[{i + 1}/{len(INCEPTION_GRID)}] {inception_label(params)}", flush=True)

    # During search, validation set is used as the evaluation split
    metrics, _ = train_and_evaluate_inception(
        params,
        d_hybrid["X_train"], d_hybrid["y_train"],
        d_hybrid["X_val"],   d_hybrid["y_val"],
        d_hybrid["X_val"],   d_hybrid["y_val"],
        class_ratio=class_ratio,
        max_epochs=30,
        patience=3
    )

    print(
        f"    Val TSS={metrics['tss']:.4f}  "
        f"F1={metrics['f1']:.4f}  "
        f"Recall={metrics['recall']:.4f}  "
        f"Train={metrics['train_time']:.1f}s\n"
    )

    if metrics["tss"] > best_tss:
        best_tss = metrics["tss"]
        best_params = copy.deepcopy(params)
        best_val_metrics = metrics

print("Best InceptionTime hyperparameters selected from hybrid validation set:")
print(f"  Params  : {inception_label(best_params)}")
print(f"  Val TSS : {best_tss:.4f}")
print(f"  Val F1  : {best_val_metrics['f1']:.4f}")


# ── Step 2: Run final experiments on all normalization variants ─────────────
print("\nRunning final InceptionTime experiments on all normalization variants...")
print("Using the same best hyperparameters selected from hybrid.\n")

for norm_key, norm_label in DATASET_NAMES.items():

    if norm_key not in datasets_inception:
        print(f"[{norm_key}] not found — skipping")
        continue

    d = datasets_inception[norm_key]

    cr = int(
        (d["y_train"] == 0).sum() /
        (d["y_train"] == 1).sum()
    )

    print("\n" + "─" * 70)
    print(f"Dataset : {norm_label}")
    print(f"Params  : {inception_label(best_params)}")
    print(f"Ratio   : {cr}:1")
    print("─" * 70)

    metrics_list = []

    for run in range(2):
        print(f"Run {run + 1}...", flush=True)

        metrics, _ = train_and_evaluate_inception(
            best_params,
            d["X_train"], d["y_train"],
            d["X_val"],   d["y_val"],
            d["X_test"],  d["y_test"],
            class_ratio=cr,
            max_epochs=30,
            patience=3
        )

        metrics_list.append(metrics)

        print(
            f"Run {run + 1} — "
            f"TSS={metrics['tss']:.4f}  "
            f"F1={metrics['f1']:.4f}  "
            f"Recall={metrics['recall']:.4f}  "
            f"Train={metrics['train_time']:.1f}s"
        )

    filepath = os.path.join(RESULTS_DIR, f"inceptiontime_{norm_key}.txt")
    save_results(metrics_list, filepath)
    print_results(metrics_list, f"InceptionTime | {norm_label}")


print("\nAll InceptionTime experiments complete.")
print(f"Results saved to: {os.path.abspath(RESULTS_DIR)}/")
print(f"Best InceptionTime params used for all variants: {inception_label(best_params)}")

════════════════════════════════════════════════════════════
  Classifier : InceptionTime
════════════════════════════════════════════════════════════

Searching 8 InceptionTime combinations on hybrid validation set only...

Hybrid train class ratio 0:1 = 104:1

[1/8] channels=8  depth=4  drop=0.1  lr=0.001  bs=128
    Val TSS=0.3511  F1=0.1069  Recall=0.4118  Train=39.2s

[2/8] channels=8  depth=4  drop=0.2  lr=0.001  bs=128
    Val TSS=0.0745  F1=0.0421  Recall=0.1176  Train=37.3s

[3/8] channels=16  depth=4  drop=0.1  lr=0.001  bs=128
    Val TSS=0.4088  F1=0.1194  Recall=0.4706  Train=64.2s

[4/8] channels=16  depth=4  drop=0.2  lr=0.001  bs=128
    Val TSS=0.1232  F1=0.0526  Recall=0.1765  Train=63.7s

[5/8] channels=16  depth=6  drop=0.1  lr=0.001  bs=128
    Val TSS=0.3904  F1=0.0664  Recall=0.5294  Train=141.7s

[6/8] channels=16  depth=6  drop=0.2  lr=0.001  bs=128
    Val TSS=0.4963  F1=0.1058  Recall=0.5882  Train=96.4s

[7/8] channels=16  depth=6  drop=0.3  lr=0.001  bs=128